In [ ]:
"""
================================================================================
MODULE 4: SPATIAL ANALYSIS OF SOCIOECONOMIC DEPRIVATION
================================================================================

Project: District-Level Multidimensional Socioeconomic Deprivation Analysis
         in Karnataka, India

Purpose: This module performs rigorous spatial econometric analysis to identify
         spatial clustering patterns, spillover effects, and hotspots of
         deprivation across Karnataka districts using REAL geographic coordinates.

Author: T SATHYA
Date: February 2026
Version: 2.0 (Real Coordinates)

================================================================================
SPATIAL ECONOMICS FOUNDATION
================================================================================

THEORETICAL MOTIVATION:
Socioeconomic deprivation is not randomly distributed in geographic space.
Neighboring districts often exhibit similar development patterns due to:

1. GEOGRAPHIC PROXIMITY: Physical accessibility and connectivity
2. INFRASTRUCTURE SPILLOVERS: Shared transportation, energy, and water networks
3. LABOR MARKET INTEGRATION: Migration and commuting patterns
4. POLICY COORDINATION: Regional development programs and governance
5. INSTITUTIONAL FACTORS: Shared historical and administrative contexts

RESEARCH CONTRIBUTION:
This spatial analysis provides HIGH NOVELTY for publication by:
- Detecting statistically significant spatial autocorrelation (Moran's I)
- Identifying deprivation clusters using Local Indicators of Spatial Association (LISA)
- Mapping hotspots and coldspots for targeted policy intervention
- Explaining spatial spillover mechanisms using real geographic data
- Informing region-based (not just district-level) policy coordination

METHODOLOGICAL RIGOR:
Unlike previous versions using simulated coordinates, this module uses:
✓ REAL latitude and longitude for all 31 Karnataka district headquarters
✓ Geodesic distance calculations for accurate spatial weights
✓ Permutation-based inference for statistical significance
✓ Publication-quality spatial econometrics following Anselin (1988, 1995)

================================================================================
REFERENCES
================================================================================

- Anselin, L. (1988). Spatial Econometrics: Methods and Models. Kluwer Academic.
- Anselin, L. (1995). Local Indicators of Spatial Association—LISA.
  Geographical Analysis, 27(2), 93-115.
- Tobler, W. (1970). A Computer Movie Simulating Urban Growth in the Detroit Region.
  Economic Geography, 46, 234-240. [First Law of Geography]
- Getis, A., & Ord, J. K. (1992). The Analysis of Spatial Association by Use of
  Distance Statistics. Geographical Analysis, 24(3), 189-206.

================================================================================
"""



from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/deprivation_analysis"




import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance_matrix
from scipy.stats import norm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.1)


# ================================================================================
# REAL DISTRICT COORDINATES FOR KARNATAKA
# ================================================================================

# District headquarters coordinates (Latitude, Longitude)
# Source: Official district headquarters from Government of Karnataka
# Coordinates verified using Google Maps and Survey of India data
# These are centroids or headquarters locations for spatial analysis

KARNATAKA_DISTRICT_COORDINATES = {
    'Bagalkot': (16.1697, 75.6947),
    'Ballari': (15.1394, 76.9214),
    'Belagavi': (15.8497, 74.4977),
    'Bengaluru Rural': (13.0747, 77.3986),
    'Bengaluru Urban': (12.9716, 77.5946),
    'Bidar': (17.9134, 77.5301),
    'Chamarajanagar': (11.9260, 76.9437),
    'Chikkaballapur': (13.4355, 77.7315),
    'Chikkamagaluru': (13.3161, 75.7720),
    'Chitradurga': (14.2226, 76.3988),
    'Dakshina Kannada': (12.8438, 74.8410),
    'Davanagere': (14.4644, 75.9218),
    'Dharwad': (15.4589, 75.0078),
    'Gadag': (15.4315, 75.6191),
    'Hassan': (13.0033, 76.1004),
    'Haveri': (14.7951, 75.4047),
    'Kalaburagi': (17.3297, 76.8343),
    'Kodagu': (12.4244, 75.7382),
    'Kolar': (13.1358, 78.1298),
    'Koppal': (15.3505, 76.1544),
    'Mandya': (12.5244, 76.8951),
    'Mysuru': (12.2958, 76.6394),
    'Raichur': (16.2120, 77.3439),
    'Ramanagara': (12.7171, 77.2800),
    'Shivamogga': (13.9299, 75.5681),
    'Tumakuru': (13.3392, 77.1006),
    'Udupi': (13.3409, 74.7421),
    'Uttara Kannada': (14.5203, 74.6861),
    'Vijayapura': (16.8302, 75.7100),
    'Yadgir': (16.7700, 77.1380),
    'Vijayanagara': (15.3500, 76.5700)  # Newly formed district
}


class SpatialDataProcessor:
    """
    Loads district data and adds real geographic coordinates for spatial analysis.
    """

    def __init__(self, filepath: str):
        """
        Initialize with path to processed data from Module 1.

        Parameters:
        -----------
        filepath : str
            Path to module1_processed_data.csv
        """
        self.filepath = filepath
        self.data = None
        self.coordinates = None

    def load_and_prepare(self) -> pd.DataFrame:
        """
        Load district data and add real geographic coordinates.

        Returns:
        --------
        pd.DataFrame
            Dataset with Latitude and Longitude columns
        """
        print("\n" + "="*80)
        print("SECTION 4.1: SPATIAL DATA LOADING AND PREPARATION")
        print("="*80)

        # Load processed data
        try:
            self.data = pd.read_csv(self.filepath)
            print(f"\n✓ Loaded data: {len(self.data)} districts")
        except FileNotFoundError:
            raise FileNotFoundError(
                f"Data file not found: {self.filepath}\n"
                f"Please ensure Module 1 has been run and output exists."
            )

        # Verify required columns
        required_cols = ['District', 'SEDI']
        missing_cols = [col for col in required_cols if col not in self.data.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")

        # Add real coordinates
        self._add_coordinates()

        # Display summary
        print(f"\n📊 Spatial Data Summary:")
        print(f"  Districts: {len(self.data)}")
        print(f"  SEDI range: [{self.data['SEDI'].min():.2f}, {self.data['SEDI'].max():.2f}]")
        print(f"  Latitude range: [{self.data['Latitude'].min():.4f}, {self.data['Latitude'].max():.4f}]")
        print(f"  Longitude range: [{self.data['Longitude'].min():.4f}, {self.data['Longitude'].max():.4f}]")

        return self.data

    def _add_coordinates(self):
        """Add real latitude and longitude to dataset."""
        print("\n🗺️  Adding Real Geographic Coordinates:")
        print("  Source: District headquarters from Government of Karnataka")
        print("  Verification: Google Maps + Survey of India")

        latitudes = []
        longitudes = []
        missing_districts = []

        for district in self.data['District']:
            if district in KARNATAKA_DISTRICT_COORDINATES:
                lat, lon = KARNATAKA_DISTRICT_COORDINATES[district]
                latitudes.append(lat)
                longitudes.append(lon)
            else:
                missing_districts.append(district)
                latitudes.append(np.nan)
                longitudes.append(np.nan)

        self.data['Latitude'] = latitudes
        self.data['Longitude'] = longitudes

        # Check for missing coordinates
        if missing_districts:
            raise ValueError(
                f"❌ Missing coordinates for {len(missing_districts)} districts:\n"
                f"{missing_districts}\n\n"
                f"Please add coordinates to KARNATAKA_DISTRICT_COORDINATES dictionary."
            )

        print(f"  ✓ Added coordinates for all {len(self.data)} districts")
        print(f"  ✓ Geographic coverage: Karnataka state boundaries")


class SpatialWeightsMatrix:
    """
    Creates spatial weights matrices using real geographic coordinates.
    """

    def __init__(self, data: pd.DataFrame):
        """
        Initialize with spatial data containing coordinates.

        Parameters:
        -----------
        data : pd.DataFrame
            Dataset with Latitude, Longitude, and SEDI columns
        """
        self.data = data.copy()
        self.coords = None
        self.dist_matrix = None
        self.W = None
        self.metadata = {}

    def create_weights(self, method: str = 'knn', k: int = 5,
                       distance_threshold: float = None) -> np.ndarray:
        """
        Create spatial weights matrix using real geographic distances.

        SPATIAL WEIGHTS THEORY:
        The weights matrix W defines neighborhood structure for spatial analysis.
        W[i,j] represents the spatial relationship between districts i and j.

        Methods:
        --------
        1. K-NEAREST NEIGHBORS (knn):
           - Each district has exactly k neighbors (closest districts)
           - Symmetric if same k for all
           - Robust to irregular spacing
           - Default: k=5 (captures local spillovers)

        2. DISTANCE THRESHOLD (distance):
           - Districts within threshold distance are neighbors
           - More realistic for policy spillovers
           - Can vary in neighbor count
           - Default threshold: median pairwise distance

        Parameters:
        -----------
        method : str
            'knn' or 'distance'
        k : int
            Number of neighbors for knn method
        distance_threshold : float
            Distance threshold in km for distance method (if None, uses median)

        Returns:
        --------
        np.ndarray
            Row-standardized spatial weights matrix (n × n)
        """
        print("\n" + "="*80)
        print("SECTION 4.2: SPATIAL WEIGHTS MATRIX CONSTRUCTION")
        print("="*80)

        print(f"\n🔗 Creating Weights Matrix:")
        print(f"  Method: {method.upper()}")
        print(f"  Using: REAL geographic coordinates (lat/lon)")

        # Extract coordinates
        self.coords = self.data[['Latitude', 'Longitude']].values
        n = len(self.coords)

        # Calculate geodesic distance matrix (in kilometers)
        self.dist_matrix = self._calculate_geodesic_distances()

        # Create binary weights matrix
        W_binary = np.zeros((n, n))

        if method == 'knn':
            # K-nearest neighbors
            for i in range(n):
                # Find k nearest neighbors (excluding self)
                distances = self.dist_matrix[i, :]
                nearest_indices = np.argsort(distances)[1:k+1]  # Exclude self (distance=0)
                W_binary[i, nearest_indices] = 1

            avg_dist = np.mean([self.dist_matrix[i, W_binary[i]==1].mean()
                               for i in range(n)])

            print(f"  ✓ K-Nearest Neighbors (k={k})")
            print(f"    Each district has {k} neighbors")
            print(f"    Average distance to neighbors: {avg_dist:.1f} km")

            self.metadata = {
                'method': 'k-nearest neighbors',
                'k': k,
                'avg_neighbor_distance_km': float(avg_dist)
            }

        elif method == 'distance':
            # Distance threshold
            if distance_threshold is None:
                # Use median of non-zero distances
                distance_threshold = np.median(self.dist_matrix[self.dist_matrix > 0])

            W_binary = (self.dist_matrix <= distance_threshold).astype(int)
            np.fill_diagonal(W_binary, 0)  # No self-neighbors

            neighbor_counts = W_binary.sum(axis=1)
            avg_neighbors = neighbor_counts.mean()

            print(f"  ✓ Distance Threshold ({distance_threshold:.1f} km)")
            print(f"    Average neighbors per district: {avg_neighbors:.1f}")
            print(f"    Min neighbors: {neighbor_counts.min()}")
            print(f"    Max neighbors: {neighbor_counts.max()}")

            self.metadata = {
                'method': 'distance threshold',
                'threshold_km': float(distance_threshold),
                'avg_neighbors': float(avg_neighbors),
                'min_neighbors': int(neighbor_counts.min()),
                'max_neighbors': int(neighbor_counts.max())
            }
        else:
            raise ValueError(f"Unknown method: {method}. Use 'knn' or 'distance'")

        # Row-standardize weights matrix
        # Each row sums to 1 → weighted average of neighbors
        row_sums = W_binary.sum(axis=1)
        self.W = np.divide(
            W_binary,
            row_sums[:, np.newaxis],
            out=np.zeros_like(W_binary, dtype=float),
            where=row_sums[:, np.newaxis] != 0
        )

        print(f"\n  ✓ Row-standardized weights matrix created")
        print(f"    Dimension: {self.W.shape[0]} × {self.W.shape[1]}")
        print(f"    Total non-zero connections: {(self.W > 0).sum()}")

        self.metadata['n_districts'] = n
        self.metadata['total_connections'] = int((self.W > 0).sum())

        return self.W

    def _calculate_geodesic_distances(self) -> np.ndarray:
        """
        Calculate geodesic (great circle) distances between all district pairs.

        Uses the Haversine formula for accurate distances on Earth's surface.

        Returns:
        --------
        np.ndarray
            Distance matrix in kilometers
        """
        print("\n  📏 Calculating geodesic distances (Haversine formula)...")

        n = len(self.coords)
        dist_matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(i+1, n):
                dist = self._haversine_distance(
                    self.coords[i, 0], self.coords[i, 1],
                    self.coords[j, 0], self.coords[j, 1]
                )
                dist_matrix[i, j] = dist
                dist_matrix[j, i] = dist

        avg_dist = dist_matrix[dist_matrix > 0].mean()
        max_dist = dist_matrix.max()

        print(f"    Average pairwise distance: {avg_dist:.1f} km")
        print(f"    Maximum distance: {max_dist:.1f} km")

        return dist_matrix

    @staticmethod
    def _haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        """
        Calculate great circle distance between two points using Haversine formula.

        Parameters:
        -----------
        lat1, lon1 : float
            Coordinates of first point (degrees)
        lat2, lon2 : float
            Coordinates of second point (degrees)

        Returns:
        --------
        float
            Distance in kilometers
        """
        # Earth's radius in kilometers
        R = 6371.0

        # Convert to radians
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)

        # Differences
        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad

        # Haversine formula
        a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        distance = R * c

        return distance


class MoransI:
    """
    Computes Global Moran's I for spatial autocorrelation testing.
    """

    def __init__(self, y: np.ndarray, W: np.ndarray, district_names: list = None):
        """
        Initialize Moran's I calculator.

        Parameters:
        -----------
        y : np.ndarray
            Variable of interest (SEDI scores)
        W : np.ndarray
            Spatial weights matrix
        district_names : list
            District names for reporting
        """
        self.y = y
        self.W = W
        self.district_names = district_names
        self.results = {}

    def calculate(self, n_permutations: int = 999) -> dict:
        """
        Calculate Global Moran's I statistic with inference.

        THEORETICAL FOUNDATION:
        Moran's I (Moran, 1950) tests for spatial autocorrelation:

        Formula:
        I = (n / Σ_i Σ_j w_ij) × [Σ_i Σ_j w_ij(y_i - ȳ)(y_j - ȳ)] / [Σ_i (y_i - ȳ)²]

        where:
        - n = number of spatial units
        - y_i = SEDI score for district i
        - ȳ = mean SEDI score
        - w_ij = spatial weight between districts i and j

        INTERPRETATION:
        - I > E[I]: Positive spatial autocorrelation (clustering of similar values)
        - I ≈ E[I]: Random spatial pattern
        - I < E[I]: Negative spatial autocorrelation (dispersion)

        E[I] = -1/(n-1) under null hypothesis of no spatial autocorrelation

        Range: Typically [-1, 1], but can exceed in small samples

        INFERENCE:
        1. Analytical: Based on asymptotic normal distribution
        2. Permutation: Non-parametric randomization test (more robust)

        Parameters:
        -----------
        n_permutations : int
            Number of permutations for inference (default: 999)

        Returns:
        --------
        dict
            Results including I statistic, p-value, z-score, interpretation
        """
        print("\n" + "="*80)
        print("SECTION 4.3: GLOBAL SPATIAL AUTOCORRELATION (MORAN'S I)")
        print("="*80)

        print("\n📊 Testing for Spatial Clustering in SEDI Scores...")

        n = len(self.y)
        y_mean = np.mean(self.y)
        y_dev = self.y - y_mean

        # Sum of all weights
        S0 = np.sum(self.W)

        # Calculate Moran's I
        numerator = 0
        for i in range(n):
            for j in range(n):
                numerator += self.W[i, j] * y_dev[i] * y_dev[j]

        denominator = np.sum(y_dev ** 2)

        I = (n / S0) * (numerator / denominator)

        # Expected value under null hypothesis
        E_I = -1 / (n - 1)

        # Variance (analytical)
        # Calculate S1 and S2
        S1 = 0.5 * np.sum((self.W + self.W.T) ** 2)
        S2 = np.sum((self.W.sum(axis=1) + self.W.sum(axis=0)) ** 2)

        b2 = (n * np.sum(y_dev ** 4)) / (np.sum(y_dev ** 2) ** 2)

        var_I = ((n * ((n**2 - 3*n + 3) * S1 - n*S2 + 3*(S0**2))) -
                 (b2 * ((n**2 - n)*S1 - 2*n*S2 + 6*(S0**2)))) / \
                ((n - 1) * (n - 2) * (n - 3) * (S0**2))

        # Z-score (analytical)
        z_score = (I - E_I) / np.sqrt(var_I)
        p_value_analytical = 2 * (1 - norm.cdf(abs(z_score)))

        # Permutation test (more robust)
        print(f"\n  🔄 Running permutation test ({n_permutations} permutations)...")

        I_permuted = []
        for _ in range(n_permutations):
            y_shuffled = np.random.permutation(self.y)
            y_dev_shuffled = y_shuffled - np.mean(y_shuffled)

            numerator_perm = 0
            for i in range(n):
                for j in range(n):
                    numerator_perm += self.W[i, j] * y_dev_shuffled[i] * y_dev_shuffled[j]

            denominator_perm = np.sum(y_dev_shuffled ** 2)
            I_perm = (n / S0) * (numerator_perm / denominator_perm)
            I_permuted.append(I_perm)

        I_permuted = np.array(I_permuted)
        p_value_permutation = (np.sum(np.abs(I_permuted) >= np.abs(I)) + 1) / (n_permutations + 1)

        # Interpretation
        if I > E_I:
            pattern = "Positive spatial autocorrelation (CLUSTERING)"
            interpretation = "Similar SEDI values cluster together spatially"
        elif I < E_I:
            pattern = "Negative spatial autocorrelation (DISPERSION)"
            interpretation = "Dissimilar SEDI values appear near each other"
        else:
            pattern = "No spatial autocorrelation (RANDOM)"
            interpretation = "SEDI values are randomly distributed in space"

        # Determine significance
        alpha = 0.05
        if p_value_permutation < alpha:
            significance = f"SIGNIFICANT (p < {alpha})"
        else:
            significance = f"NOT SIGNIFICANT (p ≥ {alpha})"

        self.results = {
            'I': float(I),
            'E_I': float(E_I),
            'variance': float(var_I),
            'z_score': float(z_score),
            'p_value_analytical': float(p_value_analytical),
            'p_value_permutation': float(p_value_permutation),
            'pattern': pattern,
            'interpretation': interpretation,
            'significance': significance,
            'n_permutations': n_permutations
        }

        # Display results
        print("\n" + "-"*80)
        print("GLOBAL MORAN'S I RESULTS")
        print("-"*80)
        print(f"  Moran's I:           {I:.4f}")
        print(f"  Expected I (null):   {E_I:.4f}")
        print(f"  Variance:            {var_I:.6f}")
        print(f"  Z-score:             {z_score:.4f}")
        print(f"  P-value (analytical):{p_value_analytical:.4f}")
        print(f"  P-value (permutation):{p_value_permutation:.4f}")
        print(f"\n  Pattern:             {pattern}")
        print(f"  Significance:        {significance}")
        print(f"\n  Interpretation:")
        print(f"    {interpretation}")

        if I > E_I and p_value_permutation < alpha:
            print(f"\n  📌 POLICY IMPLICATION:")
            print(f"    Districts with similar deprivation levels cluster together.")
            print(f"    Regional (multi-district) policies may be more effective than")
            print(f"    isolated district-level interventions.")

        return self.results


class LocalMoransI:
    """
    Computes Local Moran's I (LISA) for identifying spatial clusters.
    """

    def __init__(self, y: np.ndarray, W: np.ndarray, district_names: list):
        """
        Initialize Local Moran's I calculator.

        Parameters:
        -----------
        y : np.ndarray
            Variable of interest (SEDI scores)
        W : np.ndarray
            Spatial weights matrix
        district_names : list
            District names
        """
        self.y = y
        self.W = W
        self.district_names = district_names
        self.results = None

    def calculate(self, n_permutations: int = 999, alpha: float = 0.05) -> pd.DataFrame:
        """
        Calculate Local Moran's I (LISA) for each district.

        THEORETICAL FOUNDATION:
        Local Moran's I (Anselin, 1995) decomposes global autocorrelation into
        local contributions, identifying spatial clusters and outliers.

        Formula for district i:
        I_i = z_i × Σ_j w_ij × z_j

        where:
        - z_i = standardized SEDI score for district i
        - w_ij = spatial weight between i and j

        CLUSTER TYPES (LISA categories):

        1. HH (High-High): High SEDI surrounded by high SEDI
           → Low deprivation clusters (COLDSPOTS)
           → Districts with good socioeconomic conditions surrounded by similar

        2. LL (Low-Low): Low SEDI surrounded by low SEDI
           → High deprivation clusters (HOTSPOTS)
           → Districts with poor conditions surrounded by similar
           → PRIORITY for policy intervention

        3. HL (High-Low): High SEDI surrounded by low SEDI
           → Spatial outliers (positive outliers)
           → Well-performing districts in deprived regions

        4. LH (Low-High): Low SEDI surrounded by high SEDI
           → Spatial outliers (negative outliers)
           → Underperforming districts in developed regions

        5. NS (Not Significant): No significant local pattern

        Parameters:
        -----------
        n_permutations : int
            Number of permutations for inference
        alpha : float
            Significance level (default: 0.05)

        Returns:
        --------
        pd.DataFrame
            Local Moran's I results with cluster classifications
        """
        print("\n" + "="*80)
        print("SECTION 4.4: LOCAL SPATIAL AUTOCORRELATION (LISA)")
        print("="*80)

        print("\n🗺️  Identifying Spatial Clusters and Hotspots...")
        print(f"   Permutations: {n_permutations}")
        print(f"   Significance level: {alpha}")

        n = len(self.y)
        y_mean = np.mean(self.y)
        y_std = np.std(self.y)
        z = (self.y - y_mean) / y_std

        # Calculate Local Moran's I for each district
        I_local = np.zeros(n)
        spatial_lag = np.zeros(n)

        for i in range(n):
            # Spatial lag: weighted average of neighbors
            spatial_lag[i] = np.sum(self.W[i, :] * self.y)

            # Local Moran's I
            I_local[i] = z[i] * np.sum(self.W[i, :] * z)

        # Permutation test for each district
        print("\n  🔄 Running permutation tests...")
        p_values = np.zeros(n)

        for i in range(n):
            I_permuted = []
            for _ in range(n_permutations):
                # Shuffle all values except focal district i
                indices = list(range(n))
                indices.remove(i)
                y_shuffled = self.y.copy()
                y_shuffled[indices] = np.random.permutation(y_shuffled[indices])

                z_shuffled = (y_shuffled - np.mean(y_shuffled)) / np.std(y_shuffled)
                I_perm = z_shuffled[i] * np.sum(self.W[i, :] * z_shuffled)
                I_permuted.append(I_perm)

            I_permuted = np.array(I_permuted)
            p_values[i] = (np.sum(np.abs(I_permuted) >= np.abs(I_local[i])) + 1) / (n_permutations + 1)

        # Classify clusters (only for significant districts)
        cluster_types = []
        for i in range(n):
            if p_values[i] < alpha:
                if z[i] > 0 and spatial_lag[i] > y_mean:
                    cluster_types.append('HH')  # High-High
                elif z[i] < 0 and spatial_lag[i] < y_mean:
                    cluster_types.append('LL')  # Low-Low
                elif z[i] > 0 and spatial_lag[i] < y_mean:
                    cluster_types.append('HL')  # High-Low
                elif z[i] < 0 and spatial_lag[i] > y_mean:
                    cluster_types.append('LH')  # Low-High
                else:
                    cluster_types.append('NS')  # Edge case
            else:
                cluster_types.append('NS')  # Not significant

        # Create results dataframe
        self.results = pd.DataFrame({
            'District': self.district_names,
            'SEDI': self.y,
            'SEDI_Standardized': z,
            'Spatial_Lag': spatial_lag,
            'Local_I': I_local,
            'P_Value': p_values,
            'Significant': p_values < alpha,
            'Cluster_Type': cluster_types
        })

        # Display summary
        print("\n" + "-"*80)
        print("LOCAL MORAN'S I (LISA) RESULTS")
        print("-"*80)

        cluster_counts = self.results['Cluster_Type'].value_counts()

        print("\n  Cluster Distribution:")
        for cluster, count in cluster_counts.items():
            pct = (count / n) * 100

            if cluster == 'HH':
                desc = "High-High (Low Deprivation Clusters - COLDSPOTS)"
            elif cluster == 'LL':
                desc = "Low-Low (High Deprivation Clusters - HOTSPOTS)"
            elif cluster == 'HL':
                desc = "High-Low (Positive Outliers)"
            elif cluster == 'LH':
                desc = "Low-High (Negative Outliers)"
            else:
                desc = "Not Significant"

            print(f"    {cluster}: {count} districts ({pct:.1f}%) - {desc}")

        # Display significant clusters
        sig_results = self.results[self.results['Significant']].sort_values('Local_I', ascending=False)

        if len(sig_results) > 0:
            print(f"\n  📍 Significant Spatial Clusters (p < {alpha}):")
            print(f"     Total: {len(sig_results)} districts\n")

            for idx, row in sig_results.iterrows():
                cluster_emoji = {
                    'HH': '❄️',  # Coldspot
                    'LL': '🔥',  # Hotspot
                    'HL': '⭐',
                    'LH': '⚠️'
                }.get(row['Cluster_Type'], '○')

                print(f"    {cluster_emoji} {row['District']:20s} | SEDI: {row['SEDI']:6.2f} | "
                      f"Type: {row['Cluster_Type']} | I_local: {row['Local_I']:7.4f} | "
                      f"p-value: {row['P_Value']:.4f}")

            # Policy recommendations
            print("\n  📌 POLICY RECOMMENDATIONS:")

            ll_districts = self.results[self.results['Cluster_Type'] == 'LL']['District'].tolist()
            if ll_districts:
                print(f"\n    🔥 HIGH PRIORITY - Low-Low Clusters (Hotspots):")
                print(f"       {', '.join(ll_districts)}")
                print(f"       → These districts have low SEDI AND are surrounded by low SEDI neighbors")
                print(f"       → Recommend: Regional development programs targeting these clusters")
                print(f"       → Benefit: Spillover effects will reinforce improvements")

            hh_districts = self.results[self.results['Cluster_Type'] == 'HH']['District'].tolist()
            if hh_districts:
                print(f"\n    ❄️ BEST PRACTICES - High-High Clusters (Coldspots):")
                print(f"       {', '.join(hh_districts)}")
                print(f"       → Study policies and practices in these districts")
                print(f"       → Potential for knowledge transfer to neighboring regions")

            hl_districts = self.results[self.results['Cluster_Type'] == 'HL']['District'].tolist()
            if hl_districts:
                print(f"\n    ⭐ POSITIVE OUTLIERS - High-Low:")
                print(f"       {', '.join(hl_districts)}")
                print(f"       → Well-performing despite difficult neighborhood")
                print(f"       → Investigate success factors for replication")

            lh_districts = self.results[self.results['Cluster_Type'] == 'LH']['District'].tolist()
            if lh_districts:
                print(f"\n    ⚠️ NEGATIVE OUTLIERS - Low-High:")
                print(f"       {', '.join(lh_districts)}")
                print(f"       → Underperforming despite favorable neighborhood")
                print(f"       → Diagnose local barriers to development")
        else:
            print(f"\n  ⚠️ No significant local clusters detected at α = {alpha}")

        return self.results


class SpatialVisualizer:
    """
    Creates visualizations for spatial analysis results.
    """

    def __init__(self, data: pd.DataFrame, lisa_results: pd.DataFrame,
                 output_dir: str = f"{BASE_DIR}/output"):
        """
        Initialize visualizer.

        Parameters:
        -----------
        data : pd.DataFrame
            Spatial data with coordinates
        lisa_results : pd.DataFrame
            LISA analysis results
        output_dir : str
            Directory for saving outputs
        """
        self.data = data
        self.lisa_results = lisa_results
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def create_visualizations(self):
        """Create comprehensive spatial visualizations."""
        print("\n" + "="*80)
        print("SECTION 4.5: SPATIAL VISUALIZATION")
        print("="*80)

        # Merge LISA results with coordinate data
        viz_data = self.data.merge(
            self.lisa_results[['District', 'Cluster_Type', 'Local_I', 'Significant']],
            on='District',
            how='left'
        )

        # Create figure with multiple subplots
        fig = plt.figure(figsize=(20, 10))

        # 1. SEDI Spatial Distribution
        ax1 = plt.subplot(2, 3, 1)
        self._plot_sedi_map(viz_data, ax1)

        # 2. LISA Cluster Map
        ax2 = plt.subplot(2, 3, 2)
        self._plot_lisa_clusters(viz_data, ax2)

        # 3. Local Moran's I Values
        ax3 = plt.subplot(2, 3, 3)
        self._plot_local_i(viz_data, ax3)

        # 4. Moran Scatterplot
        ax4 = plt.subplot(2, 3, 4)
        self._plot_moran_scatterplot(viz_data, ax4)

        # 5. Cluster Distribution
        ax5 = plt.subplot(2, 3, 5)
        self._plot_cluster_distribution(viz_data, ax5)

        # 6. Geographic Coverage
        ax6 = plt.subplot(2, 3, 6)
        self._plot_geographic_coverage(viz_data, ax6)

        plt.tight_layout()

        # Save figure
        filepath = self.output_dir / 'module4_spatial_analysis_visualizations.png'
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved: module4_spatial_analysis_visualizations.png")
        plt.close()

        # Create separate detailed LISA cluster map
        self._create_detailed_cluster_map(viz_data)

    def _plot_sedi_map(self, data: pd.DataFrame, ax):
        """Plot SEDI scores on map."""
        scatter = ax.scatter(
            data['Longitude'],
            data['Latitude'],
            c=data['SEDI'],
            s=200,
            cmap='RdYlGn',
            edgecolors='black',
            linewidths=1.5,
            alpha=0.8
        )

        plt.colorbar(scatter, ax=ax, label='SEDI Score')
        ax.set_xlabel('Longitude', fontsize=11)
        ax.set_ylabel('Latitude', fontsize=11)
        ax.set_title('SEDI Spatial Distribution\n(Real Geographic Coordinates)',
                     fontsize=12, fontweight='bold')
        ax.grid(alpha=0.3)

    def _plot_lisa_clusters(self, data: pd.DataFrame, ax):
        """Plot LISA cluster classifications."""
        # Define colors for cluster types
        cluster_colors = {
            'HH': '#2ecc71',  # Green - Low deprivation clusters
            'LL': '#e74c3c',  # Red - High deprivation clusters
            'HL': '#3498db',  # Blue - Positive outliers
            'LH': '#f39c12',  # Orange - Negative outliers
            'NS': '#95a5a6'   # Gray - Not significant
        }

        for cluster_type, color in cluster_colors.items():
            cluster_data = data[data['Cluster_Type'] == cluster_type]
            if len(cluster_data) > 0:
                ax.scatter(
                    cluster_data['Longitude'],
                    cluster_data['Latitude'],
                    c=color,
                    s=200,
                    label=cluster_type,
                    edgecolors='black',
                    linewidths=1.5,
                    alpha=0.8
                )

        ax.set_xlabel('Longitude', fontsize=11)
        ax.set_ylabel('Latitude', fontsize=11)
        ax.set_title('LISA Cluster Classification\n(Local Spatial Patterns)',
                     fontsize=12, fontweight='bold')
        ax.legend(title='Cluster Type', loc='best')
        ax.grid(alpha=0.3)

    def _plot_local_i(self, data: pd.DataFrame, ax):
        """Plot Local Moran's I values."""
        scatter = ax.scatter(
            data['Longitude'],
            data['Latitude'],
            c=data['Local_I'],
            s=200,
            cmap='coolwarm',
            edgecolors='black',
            linewidths=1.5,
            alpha=0.8
        )

        plt.colorbar(scatter, ax=ax, label='Local Moran\'s I')
        ax.set_xlabel('Longitude', fontsize=11)
        ax.set_ylabel('Latitude', fontsize=11)
        ax.set_title('Local Moran\'s I Values\n(Spatial Association Strength)',
                     fontsize=12, fontweight='bold')
        ax.grid(alpha=0.3)

    def _plot_moran_scatterplot(self, data: pd.DataFrame, ax):
        """Create Moran scatterplot."""
        # Standardize SEDI
        sedi_std = (data['SEDI'] - data['SEDI'].mean()) / data['SEDI'].std()

        # Calculate spatial lag (weighted average of neighbors)
        # Use simple approach: average SEDI of nearest neighbors
        spatial_lag_std = (data['Spatial_Lag'] - data['Spatial_Lag'].mean()) / data['Spatial_Lag'].std()

        # Color by cluster type
        cluster_colors = {
            'HH': '#2ecc71', 'LL': '#e74c3c', 'HL': '#3498db',
            'LH': '#f39c12', 'NS': '#95a5a6'
        }

        for cluster_type, color in cluster_colors.items():
            mask = data['Cluster_Type'] == cluster_type
            ax.scatter(
                sedi_std[mask],
                spatial_lag_std[mask],
                c=color,
                s=100,
                label=cluster_type,
                alpha=0.7,
                edgecolors='black',
                linewidths=1
            )

        # Add quadrant lines
        ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

        # Add regression line
        z = np.polyfit(sedi_std, spatial_lag_std, 1)
        p = np.poly1d(z)
        ax.plot(sedi_std, p(sedi_std), "r-", linewidth=2, label=f'Slope: {z[0]:.3f}')

        ax.set_xlabel('SEDI (Standardized)', fontsize=11)
        ax.set_ylabel('Spatial Lag of SEDI (Standardized)', fontsize=11)
        ax.set_title('Moran Scatterplot\n(Global Spatial Association)',
                     fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=8)
        ax.grid(alpha=0.3)

    def _plot_cluster_distribution(self, data: pd.DataFrame, ax):
        """Plot cluster type distribution."""
        cluster_counts = data['Cluster_Type'].value_counts()

        colors = {
            'HH': '#2ecc71', 'LL': '#e74c3c', 'HL': '#3498db',
            'LH': '#f39c12', 'NS': '#95a5a6'
        }

        bar_colors = [colors.get(ct, '#95a5a6') for ct in cluster_counts.index]

        bars = ax.bar(range(len(cluster_counts)), cluster_counts.values, color=bar_colors, alpha=0.7)
        ax.set_xticks(range(len(cluster_counts)))
        ax.set_xticklabels(cluster_counts.index, fontsize=11)
        ax.set_ylabel('Number of Districts', fontsize=11)
        ax.set_title('Distribution of LISA Cluster Types', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for i, (bar, val) in enumerate(zip(bars, cluster_counts.values)):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

    def _plot_geographic_coverage(self, data: pd.DataFrame, ax):
        """Plot geographic coverage of Karnataka."""
        ax.scatter(
            data['Longitude'],
            data['Latitude'],
            s=100,
            c='steelblue',
            alpha=0.5,
            edgecolors='black',
            linewidths=0.5
        )

        # Add district labels for context
        for idx, row in data.iterrows():
            if row['Significant']:
                ax.annotate(
                    row['District'][:3],  # First 3 letters
                    (row['Longitude'], row['Latitude']),
                    fontsize=6,
                    ha='center',
                    alpha=0.7
                )

        ax.set_xlabel('Longitude', fontsize=11)
        ax.set_ylabel('Latitude', fontsize=11)
        ax.set_title('Karnataka Geographic Coverage\n(31 Districts)',
                     fontsize=12, fontweight='bold')
        ax.grid(alpha=0.3)

        # Add bounding box
        lat_range = data['Latitude'].max() - data['Latitude'].min()
        lon_range = data['Longitude'].max() - data['Longitude'].min()
        ax.set_xlim(data['Longitude'].min() - 0.1*lon_range,
                   data['Longitude'].max() + 0.1*lon_range)
        ax.set_ylim(data['Latitude'].min() - 0.1*lat_range,
                   data['Latitude'].max() + 0.1*lat_range)

    def _create_detailed_cluster_map(self, data: pd.DataFrame):
        """Create detailed LISA cluster map with labels."""
        fig, ax = plt.subplots(figsize=(14, 10))

        # Define colors and sizes
        cluster_colors = {
            'HH': '#2ecc71',  # Green
            'LL': '#e74c3c',  # Red
            'HL': '#3498db',  # Blue
            'LH': '#f39c12',  # Orange
            'NS': '#95a5a6'   # Gray
        }

        cluster_labels = {
            'HH': 'High-High (Low Deprivation Clusters)',
            'LL': 'Low-Low (High Deprivation Clusters - HOTSPOTS)',
            'HL': 'High-Low (Positive Outliers)',
            'LH': 'Low-High (Negative Outliers)',
            'NS': 'Not Significant'
        }

        # Plot each cluster type
        for cluster_type in ['LL', 'HH', 'HL', 'LH', 'NS']:
            cluster_data = data[data['Cluster_Type'] == cluster_type]
            if len(cluster_data) > 0:
                size = 300 if cluster_type != 'NS' else 150
                ax.scatter(
                    cluster_data['Longitude'],
                    cluster_data['Latitude'],
                    c=cluster_colors[cluster_type],
                    s=size,
                    label=cluster_labels[cluster_type],
                    edgecolors='black',
                    linewidths=2,
                    alpha=0.8,
                    zorder=3 if cluster_type != 'NS' else 1
                )

        # Add district labels (only for significant clusters)
        for idx, row in data[data['Significant']].iterrows():
            ax.annotate(
                row['District'],
                (row['Longitude'], row['Latitude']),
                fontsize=8,
                ha='center',
                va='bottom',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                         edgecolor='black', alpha=0.7),
                zorder=4
            )

        ax.set_xlabel('Longitude (°E)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Latitude (°N)', fontsize=13, fontweight='bold')
        ax.set_title('Local Indicators of Spatial Association (LISA)\n' +
                    'Karnataka Districts - Deprivation Clusters and Spatial Outliers',
                    fontsize=15, fontweight='bold', pad=20)

        ax.legend(loc='best', fontsize=10, frameon=True, fancybox=True, shadow=True)
        ax.grid(alpha=0.3, linestyle='--')

        # Improve layout
        lat_range = data['Latitude'].max() - data['Latitude'].min()
        lon_range = data['Longitude'].max() - data['Longitude'].min()
        ax.set_xlim(data['Longitude'].min() - 0.15*lon_range,
                   data['Longitude'].max() + 0.15*lon_range)
        ax.set_ylim(data['Latitude'].min() - 0.15*lat_range,
                   data['Latitude'].max() + 0.15*lat_range)

        plt.tight_layout()

        filepath = self.output_dir / 'module4_lisa_cluster_map_detailed.png'
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"✓ Saved: module4_lisa_cluster_map_detailed.png")
        plt.close()


class SpatialOutputGenerator:
    """
    Generates output files for spatial analysis results.
    """

    def __init__(self, data: pd.DataFrame, morans_results: dict,
                 lisa_results: pd.DataFrame, weights_metadata: dict,
                 output_dir: str = f"{BASE_DIR}/output"):
        """
        Initialize output generator.

        Parameters:
        -----------
        data : pd.DataFrame
            Spatial data with coordinates
        morans_results : dict
            Global Moran's I results
        lisa_results : pd.DataFrame
            LISA results
        weights_metadata : dict
            Spatial weights matrix metadata
        output_dir : str
            Output directory
        """
        self.data = data
        self.morans_results = morans_results
        self.lisa_results = lisa_results
        self.weights_metadata = weights_metadata
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def generate_outputs(self):
        """Generate all output files."""
        print("\n" + "="*80)
        print("SECTION 4.6: OUTPUT GENERATION")
        print("="*80)

        # 1. Save LISA results
        filepath = self.output_dir / 'module4_lisa_results.csv'
        self.lisa_results.to_csv(filepath, index=False)
        print(f"\n✓ Saved: module4_lisa_results.csv")

        # 2. Save spatial data with coordinates
        filepath = self.output_dir / 'module4_spatial_data.csv'
        self.data.to_csv(filepath, index=False)
        print(f"✓ Saved: module4_spatial_data.csv")

        # 3. Save comprehensive metadata
        self._save_metadata()

        # 4. Generate spatial analysis report
        self._generate_report()

        print(f"\n✅ All outputs generated successfully!")
        print(f"📁 Output directory: {self.output_dir.absolute()}")

    def _save_metadata(self):
        """Save comprehensive metadata in JSON format."""
        from datetime import datetime

        metadata = {
            'project': 'District-Level Multidimensional Socioeconomic Deprivation Analysis in Karnataka',
            'module': 'Module 4 - Spatial Analysis',
            'version': '2.0 (Real Geographic Coordinates)',
            'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'spatial_data': {
                'coordinate_source': 'Official district headquarters - Government of Karnataka',
                'coordinate_verification': 'Google Maps + Survey of India',
                'n_districts': int(len(self.data)),
                'latitude_range': [float(self.data['Latitude'].min()),
                                  float(self.data['Latitude'].max())],
                'longitude_range': [float(self.data['Longitude'].min()),
                                   float(self.data['Longitude'].max())]
            },
            'spatial_weights': self.weights_metadata,
            'global_morans_i': self.morans_results,
            'lisa_summary': {
                'total_significant': int(self.lisa_results['Significant'].sum()),
                'cluster_distribution': self.lisa_results['Cluster_Type'].value_counts().to_dict(),
                'hotspots_ll': self.lisa_results[self.lisa_results['Cluster_Type'] == 'LL']['District'].tolist(),
                'coldspots_hh': self.lisa_results[self.lisa_results['Cluster_Type'] == 'HH']['District'].tolist(),
                'outliers_hl': self.lisa_results[self.lisa_results['Cluster_Type'] == 'HL']['District'].tolist(),
                'outliers_lh': self.lisa_results[self.lisa_results['Cluster_Type'] == 'LH']['District'].tolist()
            },
            'methodology': {
                'global_autocorrelation': 'Moran\'s I with permutation inference',
                'local_autocorrelation': 'Local Moran\'s I (LISA)',
                'distance_calculation': 'Haversine formula (geodesic distances)',
                'significance_testing': 'Permutation-based randomization'
            },
            'references': [
                'Anselin, L. (1988). Spatial Econometrics: Methods and Models',
                'Anselin, L. (1995). Local Indicators of Spatial Association—LISA',
                'Tobler, W. (1970). First Law of Geography'
            ]
        }

        filepath = self.output_dir / 'module4_metadata.json'
        import json
        with open(filepath, 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f"✓ Saved: module4_metadata.json")

    def _generate_report(self):
        """Generate comprehensive spatial analysis report."""
        report = f"""
================================================================================
MODULE 4: SPATIAL ANALYSIS REPORT
================================================================================

Project: District-Level Multidimensional Socioeconomic Deprivation Analysis
Location: Karnataka, India
Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

================================================================================
1. EXECUTIVE SUMMARY
================================================================================

This spatial analysis examines the geographic distribution of socioeconomic
deprivation across {len(self.data)} districts in Karnataka using REAL district
headquarters coordinates and rigorous spatial econometric methods.

Key Findings:
- Global Moran's I: {self.morans_results['I']:.4f}
- Pattern: {self.morans_results['pattern']}
- Statistical Significance: {self.morans_results['significance']}
- Interpretation: {self.morans_results['interpretation']}

================================================================================
2. DATA SOURCES AND GEOGRAPHIC COORDINATES
================================================================================

Coordinate Source: Official district headquarters from Government of Karnataka
Verification: Google Maps + Survey of India topographic data

Geographic Coverage:
- Latitude Range: {self.data['Latitude'].min():.4f}° to {self.data['Latitude'].max():.4f}° N
- Longitude Range: {self.data['Longitude'].min():.4f}° to {self.data['Longitude'].max():.4f}° E
- Total Districts: {len(self.data)}

NOTE: Unlike previous versions using simulated coordinates, this analysis
employs REAL geographic locations, ensuring spatial validity and accuracy
for publication in peer-reviewed journals.

================================================================================
3. SPATIAL WEIGHTS MATRIX
================================================================================

Method: {self.weights_metadata['method'].upper()}
Configuration:
{self._format_weights_metadata()}

The spatial weights matrix defines neighborhood relationships for spatial
autocorrelation analysis. Row-standardized weights ensure comparability
across districts with varying neighbor counts.

================================================================================
4. GLOBAL SPATIAL AUTOCORRELATION (MORAN'S I)
================================================================================

Statistic: {self.morans_results['I']:.4f}
Expected Value (null): {self.morans_results['E_I']:.4f}
Z-Score: {self.morans_results['z_score']:.4f}
P-Value (permutation): {self.morans_results['p_value_permutation']:.4f}
Permutations: {self.morans_results['n_permutations']}

Interpretation:
{self.morans_results['interpretation']}

Statistical Significance: {self.morans_results['significance']}

Policy Implication:
{'Districts with similar deprivation levels cluster together spatially. Regional (multi-district) policies may be more effective than isolated interventions.' if self.morans_results['I'] > self.morans_results['E_I'] else 'No significant spatial clustering detected. District-specific policies may be appropriate.'}

================================================================================
5. LOCAL SPATIAL AUTOCORRELATION (LISA)
================================================================================

Total Significant Clusters: {self.lisa_results['Significant'].sum()} districts

Cluster Distribution:
{self._format_cluster_distribution()}

5.1 HIGH DEPRIVATION HOTSPOTS (Low-Low Clusters - LL)
{'----------------------------------------------------------------------' if len(self.lisa_results[self.lisa_results['Cluster_Type'] == 'LL']) > 0 else ''}
{self._format_cluster_list('LL', 'PRIORITY INTERVENTION ZONES')}

5.2 LOW DEPRIVATION COLDSPOTS (High-High Clusters - HH)
{'----------------------------------------------------------------------' if len(self.lisa_results[self.lisa_results['Cluster_Type'] == 'HH']) > 0 else ''}
{self._format_cluster_list('HH', 'BEST PRACTICE ZONES')}

5.3 POSITIVE SPATIAL OUTLIERS (High-Low - HL)
{'----------------------------------------------------------------------' if len(self.lisa_results[self.lisa_results['Cluster_Type'] == 'HL']) > 0 else ''}
{self._format_cluster_list('HL', 'SUCCESS STORIES IN CHALLENGING CONTEXTS')}

5.4 NEGATIVE SPATIAL OUTLIERS (Low-High - LH)
{'----------------------------------------------------------------------' if len(self.lisa_results[self.lisa_results['Cluster_Type'] == 'LH']) > 0 else ''}
{self._format_cluster_list('LH', 'UNDERPERFORMERS IN FAVORABLE CONTEXTS')}

================================================================================
6. POLICY RECOMMENDATIONS
================================================================================

Based on spatial cluster analysis:

1. REGIONAL COORDINATION
   - Implement multi-district development programs for spatial clusters
   - Leverage spillover effects through coordinated interventions
   - Establish regional planning authorities for clustered zones

2. TARGETED INTERVENTIONS FOR HOTSPOTS (LL Clusters)
   - Priority resource allocation to high deprivation clusters
   - Infrastructure development with regional connectivity focus
   - Capacity building programs spanning multiple districts

3. KNOWLEDGE TRANSFER FROM COLDSPOTS (HH Clusters)
   - Document and disseminate best practices
   - Establish twinning programs between high and low performers
   - Policy learning from successful regional initiatives

4. DIAGNOSTIC INVESTIGATIONS
   - Positive outliers (HL): Identify success factors for replication
   - Negative outliers (LH): Diagnose local barriers despite favorable context
   - Focus on institutional and governance factors

5. SPATIAL SPILLOVER CONSIDERATIONS
   - Account for cross-border effects in impact evaluation
   - Design policies recognizing spatial interdependencies
   - Monitor diffusion and contagion effects

================================================================================
7. METHODOLOGICAL NOTES
================================================================================

Spatial Autocorrelation Testing:
- Global: Moran's I with permutation-based inference (999 permutations)
- Local: LISA with conditional permutation for each district
- Significance level: α = 0.05

Distance Calculations:
- Method: Haversine formula (accounts for Earth's curvature)
- Unit: Kilometers
- Accuracy: Suitable for district-level analysis

Advantages of Real Coordinates (vs. Simulated):
✓ Spatial validity: True geographic relationships preserved
✓ Policy relevance: Actual distances inform intervention planning
✓ Publication quality: Meets standards for peer-reviewed journals
✓ Replicability: Results can be verified using official data sources

================================================================================
8. REFERENCES
================================================================================

Anselin, L. (1988). Spatial Econometrics: Methods and Models.
    Kluwer Academic Publishers.

Anselin, L. (1995). Local Indicators of Spatial Association—LISA.
    Geographical Analysis, 27(2), 93-115.

Tobler, W. (1970). A Computer Movie Simulating Urban Growth in the
    Detroit Region. Economic Geography, 46(sup1), 234-240.

Getis, A., & Ord, J. K. (1992). The Analysis of Spatial Association
    by Use of Distance Statistics. Geographical Analysis, 24(3), 189-206.

================================================================================
END OF REPORT
================================================================================

Generated by Module 4: Spatial Analysis
For questions or methodology details, please refer to the code documentation.
"""

        filepath = self.output_dir / 'module4_spatial_analysis_report.txt'
        with open(filepath, 'w') as f:
            f.write(report)
        print(f"✓ Saved: module4_spatial_analysis_report.txt")

    def _format_weights_metadata(self) -> str:
        """Format weights metadata for report."""
        lines = []
        for key, value in self.weights_metadata.items():
            if key != 'method':
                lines.append(f"  - {key.replace('_', ' ').title()}: {value}")
        return '\n'.join(lines)

    def _format_cluster_distribution(self) -> str:
        """Format cluster distribution for report."""
        counts = self.lisa_results['Cluster_Type'].value_counts()
        total = len(self.lisa_results)

        cluster_names = {
            'LL': 'Low-Low (High Deprivation Hotspots)',
            'HH': 'High-High (Low Deprivation Coldspots)',
            'HL': 'High-Low (Positive Outliers)',
            'LH': 'Low-High (Negative Outliers)',
            'NS': 'Not Significant'
        }

        lines = []
        for cluster in ['LL', 'HH', 'HL', 'LH', 'NS']:
            if cluster in counts.index:
                count = counts[cluster]
                pct = (count / total) * 100
                lines.append(f"  {cluster_names[cluster]:45s}: {count:2d} districts ({pct:5.1f}%)")

        return '\n'.join(lines)

    def _format_cluster_list(self, cluster_type: str, description: str) -> str:
        """Format list of districts in a cluster type."""
        districts = self.lisa_results[self.lisa_results['Cluster_Type'] == cluster_type]

        if len(districts) == 0:
            return f"None identified\n"

        lines = [f"{description}\n"]
        lines.append(f"Districts ({len(districts)}):")

        for idx, row in districts.sort_values('Local_I', ascending=False).iterrows():
            lines.append(
                f"  • {row['District']:20s} | SEDI: {row['SEDI']:6.2f} | "
                f"Local I: {row['Local_I']:7.4f} | p-value: {row['P_Value']:.4f}"
            )

        return '\n'.join(lines) + '\n'


def run_module_4(filepath: str = f"{BASE_DIR}/output/module1_processed_data.csv",
                 output_dir: str = f"{BASE_DIR}/output",
                 weights_method: str = 'knn',
                 k_neighbors: int = 5,
                 distance_threshold: float = None,
                 n_permutations: int = 999,
                 alpha: float = 0.05) -> dict:
    """
    Execute complete Module 4 spatial analysis workflow.

    Parameters:
    -----------
    filepath : str
        Path to processed data from Module 1
    output_dir : str
        Directory for output files
    weights_method : str
        Spatial weights method: 'knn' or 'distance'
    k_neighbors : int
        Number of neighbors for knn method
    distance_threshold : float
        Distance threshold in km for distance method (None = use median)
    n_permutations : int
        Number of permutations for inference
    alpha : float
        Significance level for hypothesis testing

    Returns:
    --------
    dict
        Dictionary containing all analysis results
    """
    print("\n" + "="*80)
    print("MODULE 4: SPATIAL ANALYSIS OF SOCIOECONOMIC DEPRIVATION")
    print("="*80)
    print("Project: District-Level Multidimensional Analysis - Karnataka")
    print("Version: 2.0 (Real Geographic Coordinates)")
    print("="*80)

    # Section 4.1: Load and prepare spatial data
    processor = SpatialDataProcessor(filepath)
    spatial_data = processor.load_and_prepare()

    # Section 4.2: Create spatial weights matrix
    weights_matrix = SpatialWeightsMatrix(spatial_data)
    W = weights_matrix.create_weights(
        method=weights_method,
        k=k_neighbors,
        distance_threshold=distance_threshold
    )

    # Section 4.3: Global Moran's I
    morans = MoransI(
        y=spatial_data['SEDI'].values,
        W=W,
        district_names=spatial_data['District'].tolist()
    )
    morans_results = morans.calculate(n_permutations=n_permutations)

    # Section 4.4: Local Moran's I (LISA)
    lisa = LocalMoransI(
        y=spatial_data['SEDI'].values,
        W=W,
        district_names=spatial_data['District'].tolist()
    )
    lisa_results = lisa.calculate(n_permutations=n_permutations, alpha=alpha)

    # ✅ ADD GLOBAL spatial lag (W × SEDI) to main dataframe
    spatial_data['Spatial_Lag'] = W @ spatial_data['SEDI'].values


    # Section 4.5: Create visualizations
    visualizer = SpatialVisualizer(
        data=spatial_data,
        lisa_results=lisa_results,
        output_dir=output_dir
    )
    visualizer.create_visualizations()

    # Section 4.6: Generate outputs
    output_gen = SpatialOutputGenerator(
        data=spatial_data,
        morans_results=morans_results,
        lisa_results=lisa_results,
        weights_metadata=weights_matrix.metadata,
        output_dir=output_dir
    )
    output_gen.generate_outputs()

    print("\n" + "="*80)
    print("✅ MODULE 4 COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"\nAnalyzed {len(spatial_data)} Karnataka districts")
    print(f"Global Moran's I: {morans_results['I']:.4f}")
    print(f"Significant spatial clusters: {lisa_results['Significant'].sum()}")
    print(f"Output files saved to: {output_dir}/")

    return {
        'spatial_data': spatial_data,
        'weights_matrix': W,
        'morans_results': morans_results,
        'lisa_results': lisa_results,
        'weights_metadata': weights_matrix.metadata
    }


# ================================================================================
# MAIN EXECUTION
# ================================================================================

if __name__ == "__main__":
    """
    Main execution block for Module 4.

    Usage:
    ------
    python module_4_spatial_analysis.py

    Requirements:
    -------------
    - Input file: output/module1_processed_data.csv (from Module 1)
    - Python packages: pandas, numpy, matplotlib, seaborn, scipy

    Outputs:
    --------
    - module4_spatial_data.csv: Spatial data with coordinates
    - module4_lisa_results.csv: LISA cluster classifications
    - module4_metadata.json: Comprehensive metadata
    - module4_spatial_analysis_report.txt: Detailed analysis report
    - module4_spatial_analysis_visualizations.png: Multi-panel visualization
    - module4_lisa_cluster_map_detailed.png: Detailed cluster map
    """

    try:
        # Configuration
        INPUT_FILE = f"{BASE_DIR}/output/module1_processed_data.csv"
        OUTPUT_DIR = f"{BASE_DIR}/output"
        WEIGHTS_METHOD = 'knn'  # or 'distance'
        K_NEIGHBORS = 5
        DISTANCE_THRESHOLD = None  # km (None = use median distance)
        N_PERMUTATIONS = 999
        ALPHA = 0.05

        # Run Module 4
        results = run_module_4(
            filepath=INPUT_FILE,
            output_dir=OUTPUT_DIR,
            weights_method=WEIGHTS_METHOD,
            k_neighbors=K_NEIGHBORS,
            distance_threshold=DISTANCE_THRESHOLD,
            n_permutations=N_PERMUTATIONS,
            alpha=ALPHA
        )

        # Display key results
        print("\n" + "="*80)
        print("KEY FINDINGS")
        print("="*80)
        print(f"\nGlobal Moran's I: {results['morans_results']['I']:.4f}")
        print(f"Pattern: {results['morans_results']['pattern']}")
        print(f"Significance: {results['morans_results']['significance']}")

        print("\nSpatial Clusters:")
        cluster_summary = results['lisa_results']['Cluster_Type'].value_counts()
        for cluster, count in cluster_summary.items():
            print(f"  {cluster}: {count} districts")

    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODULE 4: SPATIAL ANALYSIS OF SOCIOECONOMIC DEPRIVATION
Project: District-Level Multidimensional Analysis - Karnataka
Version: 2.0 (Real Geographic Coordinates)

SECTION 4.1: SPATIAL DATA LOADING AND PREPARATION

✓ Loaded data: 31 districts

🗺️  Adding Real Geographic Coordinates:
  Source: District headquarters from Government of Karnataka
  Verification: Google Maps + Survey of India
  ✓ Added coordinates for all 31 districts
  ✓ Geographic coverage: Karnataka state boundaries

📊 Spatial Data Summary:
  Districts: 31
  SEDI range: [0.34, 94.34]
  Latitude range: [11.9260, 17.9134]
  Longitude range: [74.4977, 78.1298]

SECTION 4.2: SPATIAL WEIGHTS MATRIX CONSTRUCTION

🔗 Creating Weights Matrix:
  Method: KNN
  Using: REAL geographic coordinates (lat/lon)

  📏 Calculating geodesic distances (Haversine formula)...
    Average pairwise distance: 267.4 km
    